In [2]:
# ==============================================================================
# File 10b: 10_notebooks/10b_1_Weaver_Substrate_Debug.ipynb
# Purpose: A sandbox to test the core Weaver-Substrate interaction.
# ==============================================================================

# %% [markdown]
# # Notebook 1: Weaver-Substrate Debugging Sandbox
# 
# This notebook tests the core mechanism of our model:
# 1. Does the `TaskWeaver` generate weights of the correct shape?
# 2. Can the `NeuralSubstrate` accept these weights and perform a forward pass without error?

# %%
import os
import sys
import torch
import yaml
from collections import OrderedDict

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from src import fabric_substrate, fabric_weaver

# %%
# Load a config file to get model parameters
config_path = 'configs/default_config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

model_config = config['model']
# Let's set some dummy values for this test
model_config['substrate']['input_dim'] = 784 # e.g. flattened MNIST
model_config['substrate']['output_dim'] = 10
model_config['weaver']['num_tasks'] = 2
model_config['weaver']['context_dim'] = 8

# %%
# 1. Initialize the Substrate
print("Initializing NeuralSubstrate...")
substrate = fabric_substrate.NeuralSubstrate(
    input_dim=model_config['substrate']['input_dim'],
    hidden_dims=model_config['substrate']['hidden_dims'],
    output_dim=model_config['substrate']['output_dim']
)
substrate_weight_shapes = substrate.get_weight_shapes()
print("\nSubstrate requires the following weight shapes:")
for name, shape in substrate_weight_shapes.items():
    print(f"  {name}: {list(shape)}")

# %%
# 2. Initialize the TaskWeaver
print("\nInitializing TaskWeaver...")
weaver = fabric_weaver.TaskWeaver(
    num_tasks=model_config['weaver']['num_tasks'],
    task_embedding_dim=model_config['weaver']['task_embedding_dim'],
    context_dim=model_config['weaver']['context_dim'],
    hidden_dim=model_config['weaver']['hidden_dim'],
    num_hidden_layers=model_config['weaver']['num_hidden_layers'],
    output_weight_shapes=substrate_weight_shapes
)

# %%
# 3. Test the generation and forward pass
print("\n--- Testing Forward Pass ---")
# Create dummy inputs
task_id = torch.tensor([0]) # Task 0
context_vector = torch.rand(model_config['weaver']['context_dim'])
input_data = torch.rand(1, model_config['substrate']['input_dim']) # Batch size of 1

print(f"Generating weights for Task ID {task_id.item()}...")
generated_weights = weaver(task_id, context_vector)

# Verify the generated weights
assert isinstance(generated_weights, OrderedDict)
assert list(generated_weights.keys()) == list(substrate_weight_shapes.keys())
print("Weaver generated a correctly structured dictionary of weights.")

print("\nPerforming forward pass on Substrate with generated weights...")
try:
    output = substrate(input_data, generated_weights)
    print("Forward pass successful!")
    print(f"Output shape: {output.shape}")
    assert output.shape == (1, model_config['substrate']['output_dim'])
    print("Output shape is correct.")
except Exception as e:
    print(f"Forward pass FAILED: {e}")



FileNotFoundError: [Errno 2] No such file or directory: 'configs/default_config.yaml'